
# 応用問題: ニューラルネットの推論 (行列積 = AI推論)

## 目標

**ニューラルネットワークの推論 (forward) の正体は「行列積 + 活性化関数」** である。これまで何度も並列化してきた行列(ベクトル)積が, そのまま AI の推論計算になっていることを体感する。

ここでは MNIST (手書き数字認識) を模した **2層 MLP** を題材にする:

- 入力 784次元 (28×28 画像を一列に並べたもの) → 隠れ層 128 (ReLU) → 出力 10クラス
- `h = ReLU(W1 x + b1)` (128次元), `o = W2 h + b2` (10次元), 予測クラス = `argmax(o)`

実データや学習は行わない。重み `W1, W2, b1, b2` と入力画像はすべて乱数 (`draw_rand01`) で生成する。予測の中身そのものに意味はないが, **計算の流れ (行列積 + ReLU + argmax) は本物**である。

## 課題

`mnist_infer.cpp` (または `mnist_infer.f90`) は, バッチ `B` 枚の画像を順に forward して予測クラスを求める。各画像の推論は互いに**独立**なので, バッチのループを並列化すればよい。

並列化すべき箇所が `TODO` になっている:

- **バッチ (画像) のループ**: 1 枚ごとに 1層目 (行列ベクトル積 + ReLU) と 2層目 (行列ベクトル積) と argmax を計算する。
  - C++: `#pragma omp parallel for reduction(+:checksum)`
  - Fortran: `!$omp parallel do private(...) reduction(+:checksum)` … `!$omp end parallel do`
    (1画像分の隠れ層 `h(128)` と出力 `o(10)` は `block` 内のローカル配列にしてあるので, スレッドごとに自動的に別物 (private) になる。)

各スレッドが別々の画像を担当するだけなので, **スレッド数を変えても予測クラスは完全に一致する** (各画像は固定された内側順序で独立に計算される)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer.exe

# Fortran
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer.exe
```

第1引数でバッチサイズ `B` (既定 64), 第2引数で繰り返し回数 `R` (既定 2000, 計測を有意にするため同じバッチを R 回推論する)。

```
OMP_NUM_THREADS=4 ./mnist_infer.exe 64 2000
```

## 期待される結果

```
batch=64, hidden=128: 予測クラス[0..7]=...,...
(結果は OMP_NUM_THREADS によらず一致する: 各画像は固定順序で独立に計算)
elapsed = ... sec
```

- **予測クラス**は 0〜9 のいずれか。`OMP_NUM_THREADS` を 1, 4, 8 と変えても**予測クラスは一致する**ことを確認せよ (これが「正しく並列化できた」基準)。
- スレッド数を増やすと `elapsed` が短くなる (台数効果)。「性能を比べる」セルで測ってみよ。
- (発展) 重い行列積を GPU にオフロードする, 内積を SIMD 化する, などで更に速くできる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mnist_infer.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (重み・入力の生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* MNIST を模した 2層 MLP の推論 (forward):
   入力 784 → 隠れ 128 (ReLU) → 出力 10。
   h = ReLU(W1 x + b1)  (128次元), o = W2 h + b2 (10次元), 予測 = argmax(o)。
   ニューラルネットの推論の正体は「行列積 + 活性化関数」であり,
   これまで並列化してきた行列(ベクトル)積がそのまま AI の推論になる。
   重みは乱数 (学習済みパラメータの代わり) なので予測の中身に意味はないが,
   計算の流れ (行列積 + ReLU + argmax) は本物である。 */

#define IN  784   /* 入力次元 */
#define HID 128   /* 隠れ層のニューロン数 */
#define OUT 10    /* 出力クラス数 */

int main(int argc, char ** argv) {
  int B = (argc > 1 ? atoi(argv[1]) : 64);     /* バッチサイズ (画像枚数) */
  int R = (argc > 2 ? atoi(argv[2]) : 2000);   /* 繰り返し回数 (計測を有意にするため) */

  /* 重み・バイアスを乱数で生成 (小さい値 [-0.05, 0.05) 付近)。 */
  double * W1 = (double *)malloc(sizeof(double) * HID * IN);
  double * b1 = (double *)malloc(sizeof(double) * HID);
  double * W2 = (double *)malloc(sizeof(double) * OUT * HID);
  double * b2 = (double *)malloc(sizeof(double) * OUT);
  for (long i = 0; i < (long)HID * IN; i++) W1[i] = (draw_rand01(i, 1) - 0.5) * 0.1;
  for (long i = 0; i < HID; i++)            b1[i] = (draw_rand01(i, 2) - 0.5) * 0.1;
  for (long i = 0; i < (long)OUT * HID; i++) W2[i] = (draw_rand01(i, 3) - 0.5) * 0.1;
  for (long i = 0; i < OUT; i++)            b2[i] = (draw_rand01(i, 4) - 0.5) * 0.1;

  /* バッチの入力「画像」B 枚 (各 784 個の数値) を乱数で生成。 */
  double * X = (double *)malloc(sizeof(double) * (long)B * IN);
  for (long i = 0; i < (long)B * IN; i++) X[i] = draw_rand01(i, 5);

  int * pred = (int *)malloc(sizeof(int) * B);
  double checksum = 0.0;

  double t0 = omp_get_wtime();
  for (int rep = 0; rep < R; rep++) {
    checksum = 0.0;
    /* 各画像の forward は互いに独立。バッチを分担して並列に推論する。 */
    // TODO: バッチ (画像) のループを #pragma omp parallel for reduction(+:checksum) で並列化せよ.
    for (int n = 0; n < B; n++) {
      const double * x = X + (long)n * IN;
      double h[HID];
      double o[OUT];
      /* 1層目: h = ReLU(W1 x + b1)  (行列ベクトル積 + ReLU) */
      for (int j = 0; j < HID; j++) {
        double s = b1[j];
        for (int k = 0; k < IN; k++) s += W1[(long)j * IN + k] * x[k];
        h[j] = (s > 0.0 ? s : 0.0);   /* ReLU */
      }
      /* 2層目: o = W2 h + b2  (行列ベクトル積) */
      for (int c = 0; c < OUT; c++) {
        double s = b2[c];
        for (int k = 0; k < HID; k++) s += W2[(long)c * HID + k] * h[k];
        o[c] = s;
        checksum += s;
      }
      /* 予測クラス = argmax(o) */
      int amax = 0;
      for (int c = 1; c < OUT; c++) if (o[c] > o[amax]) amax = c;
      pred[n] = amax;
    }
  }
  double elapsed = omp_get_wtime() - t0;

  int show = (B < 8 ? B : 8);
  printf("batch=%d, hidden=%d: 予測クラス[0..%d]=", B, HID, show - 1);
  for (int n = 0; n < show; n++) printf("%d%s", pred[n], n + 1 < show ? "," : "");
  printf(", checksum=%.6f\n", checksum);
  printf("(結果は OMP_NUM_THREADS によらず一致する: 各画像は固定順序で独立に計算)\n");
  printf("elapsed = %.3f sec\n", elapsed);
  free(W1); free(b1); free(W2); free(b2); free(X); free(pred);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mnist_infer.f90
module mlp_mod
  integer, parameter :: IN = 784   ! 入力次元
  integer, parameter :: HID = 128  ! 隠れ層のニューロン数
  integer, parameter :: OUTC = 10  ! 出力クラス数
contains
  ! 状態を持たない乱数 (重み・入力の生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01
end module mlp_mod

! MNIST を模した 2層 MLP の推論 (forward):
! 入力 784 → 隠れ 128 (ReLU) → 出力 10。
! h = ReLU(W1 x + b1) (128次元), o = W2 h + b2 (10次元), 予測 = argmax(o)。
! ニューラルネットの推論の正体は「行列積 + 活性化関数」であり,
! これまで並列化してきた行列(ベクトル)積がそのまま AI の推論になる。
! 重みは乱数 (学習済みパラメータの代わり) なので予測の中身に意味はないが,
! 計算の流れ (行列積 + ReLU + argmax) は本物である。
program mnist_infer
  use mlp_mod
  use omp_lib
  character(len=32) :: arg
  integer :: B, R, rep, n, j, c, k, amax, show
  real(8) :: s, checksum, t0, elapsed
  real(8), allocatable :: W1(:), b1(:), W2(:), b2(:), X(:)
  integer, allocatable :: pred(:)
  integer(8) :: i8

  B = 64; R = 2000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) B
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) R
  end if

  ! 重み・バイアスを乱数で生成 (小さい値 [-0.05, 0.05) 付近)。添字は 0 始まり。
  allocate(W1(0:HID*IN-1), b1(0:HID-1), W2(0:OUTC*HID-1), b2(0:OUTC-1))
  allocate(X(0:int(B,8)*IN-1), pred(0:B-1))
  do i8 = 0, int(HID,8)*IN - 1
     W1(i8) = (draw_rand01(i8, 1_8) - 0.5d0) * 0.1d0
  end do
  do i8 = 0, HID - 1
     b1(i8) = (draw_rand01(i8, 2_8) - 0.5d0) * 0.1d0
  end do
  do i8 = 0, int(OUTC,8)*HID - 1
     W2(i8) = (draw_rand01(i8, 3_8) - 0.5d0) * 0.1d0
  end do
  do i8 = 0, OUTC - 1
     b2(i8) = (draw_rand01(i8, 4_8) - 0.5d0) * 0.1d0
  end do
  do i8 = 0, int(B,8)*IN - 1
     X(i8) = draw_rand01(i8, 5_8)
  end do

  t0 = omp_get_wtime()
  do rep = 1, R
     checksum = 0.0d0
     ! 各画像の forward は互いに独立。バッチを分担して並列に推論する。
     ! TODO: バッチ (画像) のループを !$omp parallel do で並列化せよ (per-image の作業を block 内のローカル配列で行い自動的に private にする).
     do n = 0, B - 1
        block
          real(8) :: h(0:HID-1), o(0:OUTC-1)
          ! 1層目: h = ReLU(W1 x + b1) (行列ベクトル積 + ReLU)
          do j = 0, HID - 1
             s = b1(j)
             do k = 0, IN - 1
                s = s + W1(int(j,8)*IN + k) * X(int(n,8)*IN + k)
             end do
             if (s > 0.0d0) then
                h(j) = s
             else
                h(j) = 0.0d0
             end if
          end do
          ! 2層目: o = W2 h + b2 (行列ベクトル積)
          do c = 0, OUTC - 1
             s = b2(c)
             do k = 0, HID - 1
                s = s + W2(int(c,8)*HID + k) * h(k)
             end do
             o(c) = s
             checksum = checksum + s
          end do
          ! 予測クラス = argmax(o)
          amax = 0
          do c = 1, OUTC - 1
             if (o(c) > o(amax)) amax = c
          end do
          pred(n) = amax
        end block
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end do
  elapsed = omp_get_wtime() - t0

  show = min(B, 8)
  write (*, '(a,i0,a,i0,a,i0,a)', advance='no') &
       "batch=", B, ", hidden=", HID, ": 予測クラス[0..", show-1, "]="
  do n = 0, show - 1
     if (n < show - 1) then
        write (*, '(i0,a)', advance='no') pred(n), ","
     else
        write (*, '(i0)', advance='no') pred(n)
     end if
  end do
  write (*, '(a,f0.6)') ", checksum=", checksum
  print "(a)", "(結果は OMP_NUM_THREADS によらず一致する: 各画像は固定順序で独立に計算)"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program mnist_infer

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mnist_infer.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mnist_infer.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mnist_infer.cpp}